# Bước 06_4: Thử Nghiệm Ablation Study Đánh Giá Mức Độ Phụ Thuộc Vào Đặc Trưng Lag / Rolling
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers
Nguồn tham chiếu: `srcs/05_machine_learning/Forcasting_v3/`

## 1. Tổng quan & Mục đích Thử nghiệm
Khi phân tích chuỗi thời gian sản lượng quang điện, đặc trưng `lag_1` (sản lượng 15 phút trước) có Mutual Information cực cao (~1.37) và đóng vai trò chi phối. Điều này khiến mô hình dễ gặp hiện tượng **Trễ pha (Lagging effect)** — tức dự báo chỉ đơn thuần "bám theo" mốc $t-1$.

### 4 Biến thể đặc trưng được thử nghiệm:
1. **`full` (Biến thể A):** Sử dụng đầy đủ bộ đặc trưng (bao gồm lag, rolling, thời tiết, thời gian, metadata).
2. **`no_lag1` (Biến thể B):** Bỏ riêng đặc trưng `lag_1`.
3. **`no_lag` (Biến thể C):** Bỏ toàn bộ các đặc trưng bắt đầu bằng `lag_`.
4. **`no_lag_rolling` (Biến thể D):** Bỏ toàn bộ các đặc trưng `lag_` VÀ `rolling_` (chỉ còn thời tiết + thời gian + metadata trạm).

> ### Quy tắc siêu tham số & Optuna Tuning:
> - **Biến thể A, B, C:** Tái sử dụng tham số tối ưu `best_params` của loss chiến thắng (đọc từ `best_loss.json` và `best_params.json`). Việc không tune lại giúp so sánh công bằng tác động của việc bỏ đặc trưng.
> - **Biến thể D (`no_lag_rolling`):** BẮT BUỘC TUNE LẠI BẰNG OPTUNA (`N_TRIALS = 15`). Lý do: Khi loại bỏ hoàn toàn nhóm lag và rolling, không gian đặc trưng thay đổi bản chất, việc dùng lại tham số cũ sẽ làm biến thể D bị thiệt thòi không công bằng.

## 2. Import thư viện và khai báo tham số

In [ ]:
import gc
import json
import os
import pickle
import platform
import statistics
import time
import warnings

import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import pyarrow.parquet as pq
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Tham số chung ──
VERSION = 'v3'
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'
HORIZONS = [1, 4]
FOLDS = [1, 2, 3, 4, 5]
N_TRIALS_D = 15
SEED = 42
EARLY_STOPPING_ROUNDS = 100

USE_GPU = True
GPU_PLATFORM_ID = 0
GPU_DEVICE_ID = 0

# ── Thư mục dữ liệu ──
SELECTED_DIR = '../../data/model/v3/05_selected'
TRAIN_BASE_DIR = '../../data/model/v3/06_train'
TEST_FINAL_DIR = '../../data/model/v3/07_final_test'
BASELINE_DIR = '../../data/model/v3/06_0_baseline'
OUTPUT_DIR = '../../data/model/v3/06_4_ablation'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Đã import thư viện và khai báo tham số cho Notebook 06_4.")
print("Thư mục xuất kết quả:", OUTPUT_DIR)

## 3. Thiết lập GPU (OpenCL) cho LightGBM

In [ ]:
LA_LINUX = (os.name == "posix" and platform.system() == "Linux")

if LA_LINUX:
    OCL_CANDIDATES = [
        "/run/opengl-driver/etc/OpenCL/vendors",
        "/etc/OpenCL/vendors",
    ]
    if "OCL_ICD_VENDORS" not in os.environ:
        for _p in OCL_CANDIDATES:
            if os.path.isdir(_p) and any(f.endswith(".icd") for f in os.listdir(_p)):
                os.environ["OCL_ICD_VENDORS"] = _p
                print("Đã tự đặt OCL_ICD_VENDORS = " + str(_p))
                break


def kiem_tra_gpu():
    try:
        X = np.random.rand(100, 4)
        y = np.random.rand(100)
        lgb.train(
            {"objective": "regression", "device": "gpu", "gpu_platform_id": GPU_PLATFORM_ID, "gpu_device_id": GPU_DEVICE_ID, "verbose": -1},
            lgb.Dataset(X, y),
            num_boost_round=1
        )
        return True, ""
    except Exception as e:
        return False, str(e)[:200]

GPU_SAN_SANG = False
if USE_GPU:
    GPU_SAN_SANG, _err = kiem_tra_gpu()
    if GPU_SAN_SANG:
        print("GPU OpenCL sẵn sàng. LightGBM sẽ chạy trên GPU.")
    else:
        print("[CẢNH BÁO] Không dùng được GPU, tự động chuyển sang CPU.")
        print("Lý do: " + str(_err))

print("Chế độ tính toán chính thức: " + ("GPU" if GPU_SAN_SANG else "CPU"))

## 4. Đọc loss thắng và cấu hình đặc trưng ban đầu

In [ ]:
# 1. Đọc loss thắng từ best_loss.json
best_loss_path = f'{TEST_FINAL_DIR}/best_loss.json'
winning_loss = 'huber'

if os.path.exists(best_loss_path):
    with open(best_loss_path, 'r', encoding='utf-8') as f:
        bl_data = json.load(f)
    if 'h1' in bl_data:
        winning_loss = bl_data['h1']['winning_loss']
    elif 'winning_loss' in bl_data:
        winning_loss = bl_data['winning_loss']

print("Loss chiến thắng được sử dụng làm baseline: " + str(winning_loss.upper()))

# 2. Đọc selected_features.json
json_path = f'{SELECTED_DIR}/selected_features.json'
if not os.path.exists(json_path):
    raise FileNotFoundError("Không tìm thấy selected_features.json tại: " + json_path)

with open(json_path, 'r', encoding='utf-8') as f:
    _sel_raw = json.load(f)

selected_features = _sel_raw['selected_features'] if isinstance(_sel_raw, dict) else _sel_raw

NEEDED_COLS = selected_features + [
    TARGET_COL, SITE_COL, TIMESTAMP_COL,
    "energy_source", "exclude_from_training",
    "outlier_group", "has_complete_history_features", "is_daylight"
]


def read_selected(path):
    have = set(pq.ParquetFile(path).schema_arrow.names)
    cols = [c for c in NEEDED_COLS if c in have]
    return pd.read_parquet(path, columns=cols)


def add_horizon_target(df, horizon_steps):
    out = df.copy()
    target_col_name = f'target_h{horizon_steps}'
    if horizon_steps == 1:
        out[target_col_name] = out[TARGET_COL]
    else:
        shift_steps = -(horizon_steps - 1)
        out[target_col_name] = out.groupby(SITE_COL)[TARGET_COL].shift(shift_steps)
    return out, target_col_name


def filter_valid_rows(df, name="", target_col=TARGET_COL):
    n_before = len(df)
    out = df.copy()

    if 'exclude_from_training' in out.columns:
        out = out[out['exclude_from_training'] == False]

    if 'has_complete_history_features' in out.columns:
        out = out[out['has_complete_history_features'] == True]

    if target_col in out.columns:
        out = out[out[target_col].notna()]

    n_after = len(out)
    n_removed = n_before - n_after
    pct = (n_after / n_before) * 100 if n_before > 0 else 0
    print("Lọc dữ liệu " + str(name) + ": ban đầu " + str(n_before) + " dòng -> còn " + str(n_after) + " dòng (loại " + str(n_removed) + " dòng, " + str(round(pct, 2)) + "%)")
    return out


print("Số đặc trưng ban đầu (full): " + str(len(selected_features)))

## 5. Định nghĩa 4 biến thể và kiểm tra siêu tham số

In [ ]:
def get_variant_features(variant_name, base_features):
    if variant_name == "full":
        return list(base_features)
    elif variant_name == "no_lag1":
        return [f for f in base_features if f != "lag_1"]
    elif variant_name == "no_lag":
        return [f for f in base_features if not f.startswith("lag_")]
    elif variant_name == "no_lag_rolling":
        return [f for f in base_features if not f.startswith("lag_") and not f.startswith("rolling_")]
    else:
        raise ValueError("Unknown variant: " + str(variant_name))

VARIANTS = ["full", "no_lag1", "no_lag", "no_lag_rolling"]

# Kiểm tra sự tồn tại của best_params.json cho A, B, C
best_params_dict = {}
for h in HORIZONS:
    bp_path = f'{TRAIN_BASE_DIR}/{winning_loss}/h{h}/best_params.json'
    if not os.path.exists(bp_path):
        bp_path = f'{TRAIN_BASE_DIR}/{winning_loss}/best_params.json'

    if not os.path.exists(bp_path):
        raise FileNotFoundError(
            "Không tìm thấy best_params.json tại: " + bp_path + ". "
            "Hãy chạy notebook 06_" + winning_loss + " trước!"
        )

    with open(bp_path, 'r', encoding='utf-8') as f:
        best_params_dict[f"h{h}"] = json.load(f)

print("Đã xác nhận sự tồn tại của best_params.json cho cả h1 và h4.")
for v in VARIANTS:
    feats = get_variant_features(v, selected_features)
    print("Biến thể [" + str(v) + "]: " + str(len(feats)) + " đặc trưng")

## 6. Chạy thử nghiệm trọn vẹn 4 biến thể Ablation

In [ ]:
ablation_metrics_rows = []
predictions_sample_dict = {}  # Lưu chuỗi dự báo mẫu để vẽ biểu đồ so sánh

for v_name in VARIANTS:
    v_dir = f'{OUTPUT_DIR}/{v_name}'
    os.makedirs(v_dir, exist_ok=True)

    for h in HORIZONS:
        h_label = f"h{h}"
        t0_var = time.time()
        v_features = get_variant_features(v_name, selected_features)

        print("")
        print("=" * 80)
        print("=== BẮT ĐẦU CHẠY BIẾN THỂ [" + str(v_name.upper()) + "] | HORIZON " + str(h_label.upper()) + " | SỐ FEATURE: " + str(len(v_features)) + " ===")
        print("=" * 80)

        # Nạp tập development
        dev_path = f'{SELECTED_DIR}/{VERSION}_development_selected.parquet'
        dev_raw = read_selected(dev_path)
        dev_raw, t_col = add_horizon_target(dev_raw, h)
        dev_df = filter_valid_rows(dev_raw, f"Dev {v_name} {h_label}", target_col=t_col)
        del dev_raw
        gc.collect()

        feat_cols = [c for c in v_features if c in dev_df.columns]
        num_cols = [c for c in feat_cols if pd.api.types.is_numeric_dtype(dev_df[c])]
        feat_cols = num_cols
        cat_cols = [c for c in feat_cols if c.endswith('_enc')]

        medians = dev_df[feat_cols].median(numeric_only=True).fillna(0.0)
        X_dev = dev_df[feat_cols].fillna(medians).astype(np.float32)
        y_dev = dev_df[t_col].astype(np.float32)

        # Xác định tham số huấn luyện
        if v_name == "no_lag_rolling":
            print("Optuna Tuning cho Biến thể D (no_lag_rolling)...")
            # Cache 5 fold để tune Optuna cho D
            cached_folds = []
            for fold in FOLDS:
                tr_p = f'{SELECTED_DIR}/time_series_folds/fold_{fold}_train_selected.parquet'
                va_p = f'{SELECTED_DIR}/time_series_folds/fold_{fold}_val_selected.parquet'
                if os.path.exists(tr_p) and os.path.exists(va_p):
                    tr_r, tc = add_horizon_target(read_selected(tr_p), h)
                    va_r, tc = add_horizon_target(read_selected(va_p), h)
                    tr_f = filter_valid_rows(tr_r, f"Fold {fold} Tr", target_col=tc)
                    va_f = filter_valid_rows(va_r, f"Fold {fold} Va", target_col=tc)
                    del tr_r, va_r

                    c_feats = [c for c in feat_cols if c in tr_f.columns]
                    m_f = tr_f[c_feats].median(numeric_only=True).fillna(0.0)
                    cached_folds.append({
                        'x_tr': tr_f[c_feats].fillna(m_f).astype(np.float32),
                        'y_tr': tr_f[tc].astype(np.float32),
                        'x_va': va_f[c_feats].fillna(m_f).astype(np.float32),
                        'y_va': va_f[tc].astype(np.float32),
                        'cat': [c for c in c_feats if c.endswith('_enc')]
                    })
                    del tr_f, va_f
                    gc.collect()

            def obj_d(trial):
                reg_t = trial.suggest_categorical("reg_type", ["l1", "l2", "elasticnet"])
                p_dict = {
                    'objective': 'huber' if winning_loss == 'huber' else ('regression_l1' if winning_loss == 'mae' else 'regression'),
                    'n_estimators': trial.suggest_int("n_estimators", 200, 800),
                    'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
                    'num_leaves': trial.suggest_int("num_leaves", 31, 127),
                    'min_child_samples': trial.suggest_int("min_child_samples", 20, 200),
                    'subsample': trial.suggest_float("subsample", 0.7, 1.0),
                    'colsample_bytree': trial.suggest_float("colsample_bytree", 0.7, 1.0),
                    'reg_alpha': trial.suggest_float("reg_alpha", 0.0, 10.0) if reg_t in ["l1", "elasticnet"] else 0.0,
                    'reg_lambda': trial.suggest_float("reg_lambda", 0.0, 10.0) if reg_t in ["l2", "elasticnet"] else 0.0,
                    'random_state': SEED, 'n_jobs': -1, 'verbosity': -1
                }
                if GPU_SAN_SANG:
                    p_dict['device'] = 'gpu'
                    p_dict['gpu_platform_id'] = GPU_PLATFORM_ID
                    p_dict['gpu_device_id'] = GPU_DEVICE_ID

                err_s, y_s = 0.0, 0.0
                for fp in cached_folds:
                    m = LGBMRegressor(**p_dict)
                    try:
                        m.fit(fp['x_tr'], fp['y_tr'], eval_set=[(fp['x_va'], fp['y_va'])], eval_metric='l1', callbacks=[early_stopping(100, verbose=False)])
                    except Exception:
                        p_cpu = p_dict.copy()
                        p_cpu['device'] = 'cpu'
                        m = LGBMRegressor(**p_cpu)
                        m.fit(fp['x_tr'], fp['y_tr'], eval_set=[(fp['x_va'], fp['y_va'])], eval_metric='l1', callbacks=[early_stopping(100, verbose=False)])
                    pr = m.predict(fp['x_va'])
                    yv = fp['y_va'].to_numpy(dtype=float)
                    err_s += float(np.nansum(np.abs(yv - pr)))
                    y_s += float(np.nansum(np.abs(yv)))
                return (err_s / y_s * 100.0) if y_s > 0 else float("inf")

            study_d = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED), pruner=MedianPruner())
            study_d.optimize(obj_d, n_trials=N_TRIALS_D, show_progress_bar=False)

            best_p_d = study_d.best_params.copy()
            reg_t = best_p_d.pop('reg_type', None)
            final_n_est = best_p_d.pop('n_estimators', 500)
            reg_alpha = best_p_d.pop('reg_alpha', 0.0)
            reg_lambda = best_p_d.pop('reg_lambda', 0.0)

            final_params = {
                'objective': 'huber' if winning_loss == 'huber' else ('regression_l1' if winning_loss == 'mae' else 'regression'),
                'n_estimators': final_n_est,
                'reg_alpha': reg_alpha,
                'reg_lambda': reg_lambda,
                'random_state': SEED, 'n_jobs': -1, 'verbosity': -1,
                **best_p_d
            }
            # Ghi file best_params cho D
            with open(f'{OUTPUT_DIR}/best_params_no_lag_rolling.json', 'w', encoding='utf-8') as f:
                json.dump({'horizon': h_label, 'best_wape': study_d.best_value, 'params': final_params}, f, indent=2)

            del cached_folds
            gc.collect()
        else:
            # Re-use best_params tu loss thang cho A, B, C
            bp_loaded = best_params_dict[h_label]
            final_n_est = bp_loaded.get('final_n_estimators', 500)
            b_params = bp_loaded.get('best_params', {}).copy()
            b_params.pop('reg_type', None)
            b_params.pop('n_estimators', None)
            reg_alpha = b_params.pop('reg_alpha', 0.0)
            reg_lambda = b_params.pop('reg_lambda', 0.0)

            final_params = {
                'objective': 'huber' if winning_loss == 'huber' else ('regression_l1' if winning_loss == 'mae' else 'regression'),
                'n_estimators': final_n_est,
                'reg_alpha': reg_alpha,
                'reg_lambda': reg_lambda,
                'random_state': SEED, 'n_jobs': -1, 'verbosity': -1,
                **b_params
            }

        if GPU_SAN_SANG:
            final_params['device'] = 'gpu'
            final_params['gpu_platform_id'] = GPU_PLATFORM_ID
            final_params['gpu_device_id'] = GPU_DEVICE_ID

        # Train Final Model cho biến thể v_name
        model_v = LGBMRegressor(**final_params)
        try:
            model_v.fit(X_dev, y_dev, categorical_feature=cat_cols if cat_cols else 'auto')
        except Exception:
            cp = final_params.copy()
            cp['device'] = 'cpu'
            model_v = LGBMRegressor(**cp)
            model_v.fit(X_dev, y_dev, categorical_feature=cat_cols if cat_cols else 'auto')

        # Save model pkl
        with open(f'{v_dir}/model_{h_label}.pkl', 'wb') as f:
            pickle.dump(model_v, f)

        del dev_df, X_dev, y_dev
        gc.collect()

        # Nạp tập Validation để đánh giá
        val_path = f'{SELECTED_DIR}/{VERSION}_val_selected.parquet'
        val_raw = read_selected(val_path)
        val_raw, t_col = add_horizon_target(val_raw, h)
        val_df = filter_valid_rows(val_raw, f"Val {v_name} {h_label}", target_col=t_col)
        del val_raw
        gc.collect()

        X_val = val_df[feat_cols].fillna(medians).astype(float)
        y_true = val_df[t_col].values
        y_pred = model_v.predict(X_val)

        # Lưu lại dự báo của 1 trạm mẫu để vẽ biểu đồ ở cell sau
        if h == 1:
            sample_site_id = val_df[SITE_COL].iloc[0]
            site_mask = val_df[SITE_COL] == sample_site_id
            if 'y_true_sample' not in predictions_sample_dict:
                predictions_sample_dict['timestamp'] = val_df.loc[site_mask, TIMESTAMP_COL].values[:150]
                predictions_sample_dict['y_true'] = y_true[site_mask][:150]
            predictions_sample_dict[f'pred_{v_name}'] = y_pred[site_mask][:150]

        def compute_wape_f(yt, yp):
            abs_y = np.nansum(np.abs(yt))
            return (np.nansum(np.abs(yt - yp)) / abs_y * 100.0) if abs_y > 0 else np.nan

        def compute_metrics_f(yt, yp):
            return {
                'wape': compute_wape_f(yt, yp),
                'rmse': root_mean_squared_error(yt, yp),
                'mae': mean_absolute_error(yt, yp),
                'r2': r2_score(yt, yp),
            }

        mask_all = np.ones(len(val_df), dtype=bool)
        mask_meas = (val_df['energy_source'] == 'measured').values if 'energy_source' in val_df.columns else mask_all
        mask_day = ((val_df['is_daylight'] == True).values | (val_df['is_daylight'] == 1).values) if 'is_daylight' in val_df.columns else mask_meas
        mask_meas_day = mask_meas & mask_day

        scopes = {'all': mask_all, 'measured': mask_meas, 'measured_daylight': mask_meas_day}

        for s_name, s_mask in scopes.items():
            m = compute_metrics_f(y_true[s_mask], y_pred[s_mask])
            ablation_metrics_rows.append({
                'variant': v_name,
                'horizon': h_label,
                'horizon_steps': h,
                'scope': s_name,
                'num_features': len(feat_cols),
                'total_rows': int(s_mask.sum()),
                **m
            })

        t_elapsed = time.time() - t0_var
        m_head = compute_metrics_f(y_true[mask_meas_day], y_pred[mask_meas_day])
        print("Hoàn tất [" + str(v_name.upper()) + "] " + str(h_label.upper()) + ": WAPE Measured&Daylight = " + str(round(m_head['wape'], 2)) + "% | Thời gian: " + str(round(t_elapsed, 1)) + "s")

        del val_df, X_val, y_true, y_pred
        gc.collect()

df_ablation = pd.DataFrame(ablation_metrics_rows)
print("")
print("Đã hoàn thành thử nghiệm 4 biến thể Ablation cho cả h1 và h4.")

## 7. Bảng so sánh kết quả chính và kết luận tự động

In [ ]:
# Nạp WAPE baseline persistence từ 06_0_baseline
base_path = f'{BASELINE_DIR}/baseline_metrics.csv'
persistence_wape_dict = {}

if os.path.exists(base_path):
    df_b = pd.read_csv(base_path)
    # Lấy persistence_current cho measured_daylight
    p_row = df_b[(df_b['baseline_model'] == 'persistence_current') & (df_b['scope'] == 'measured_daylight')]
    for _, r in p_row.iterrows():
        persistence_wape_dict[r['horizon']] = r['wape']

# Lập bảng so sánh chính ở phạm vi tiêu chuẩn measured_daylight
main_table = df_ablation[df_ablation['scope'] == 'measured_daylight'].copy()

# Lấy WAPE của biến thể A (full) làm mốc chênh lệch
wape_full_map = main_table[main_table['variant'] == 'full'].set_index('horizon')['wape'].to_dict()

main_table['wape_diff_vs_variant_A'] = main_table.apply(
    lambda r: r['wape'] - wape_full_map.get(r['horizon'], r['wape']), axis=1
)
main_table['persistence_wape'] = main_table['horizon'].map(persistence_wape_dict)
main_table['vs_persistence_diff'] = main_table['wape'] - main_table['persistence_wape']

print("=== BẢNG KẾT QUẢ ABLATION STUDY CHÍNH (PHẠM VI MEASURED & DAYLIGHT) ===")
display_cols = ['horizon', 'variant', 'num_features', 'wape', 'rmse', 'r2', 'wape_diff_vs_variant_A', 'persistence_wape', 'vs_persistence_diff']
display(main_table[display_cols])

print("")
print("=== KẾT LUẬN TỰ ĐỘNG TỪ SỐ LIỆU THỬ NGHIỆM ===")

wape_A_h1 = wape_full_map.get('h1', 0)
wape_B_h1 = main_table[(main_table['variant'] == 'no_lag1') & (main_table['horizon'] == 'h1')]['wape'].values[0]
wape_D_h1 = main_table[(main_table['variant'] == 'no_lag_rolling') & (main_table['horizon'] == 'h1')]['wape'].values[0]
pers_h1 = persistence_wape_dict.get('h1', 15.0)

print(f"1. Khi bỏ riêng lag_1 (Biến thể B): WAPE tăng từ {wape_A_h1:.2f}% lên {wape_B_h1:.2f}% (tăng {wape_B_h1 - wape_A_h1:+.2f}%).")
print(f"2. Khi bỏ toàn bộ Lag & Rolling (Biến thể D): WAPE tăng lên {wape_D_h1:.2f}% (Persistence Baseline là {pers_h1:.2f}%).")

if wape_B_h1 >= pers_h1 - 1.0:
    print("-> ĐÁNH GIÁ: Mô hình phụ thuộc cực kỳ mạnh vào lag_1. Khi mất lag_1, chất lượng dự báo giảm sâu tiệm cận Baseline Persistence.")
else:
    print("-> ĐÁNH GIÁ: Mô hình vẫn giữ được khoảng cách hiệu quả so với Baseline Persistence nhờ các đặc trưng thời tiết và thời gian.")

## 8. Trực quan hóa kết quả Ablation Study

In [ ]:
print("--- TRỰC QUAN HÓA KẾT QUẢ ABLATION STUDY ---")

# 1. Bar chart WAPE 4 biến thể + Đường ngang Baseline Persistence
fig, ax = plt.subplots(figsize=(10, 5))
main_h1 = main_table[main_table['horizon'] == 'h1']

bars = ax.bar(main_h1['variant'], main_h1['wape'], color=['navy', 'royalblue', 'darkorange', 'firebrick'], alpha=0.85)
if 'h1' in persistence_wape_dict:
    p_val = persistence_wape_dict['h1']
    ax.axhline(p_val, color='black', linestyle='--', linewidth=2, label=f'Persistence Baseline ({p_val:.2f}%)')

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.2, f'{yval:.2f}%', ha='center', va='bottom', fontweight='bold')

ax.set_title('So sánh Sai số WAPE giữa 4 Biến thể Ablation (h1: t+1)')
ax.set_ylabel('WAPE (%)')
ax.legend()
plt.tight_layout()
plt.show()

# 2. Biểu đồ đường Plotly tương tác vẽ chồng 4 biến thể vs Thực tế
if 'y_true' in predictions_sample_dict:
    print("Đang vẽ biểu đồ đường tương tác Plotly so sánh độ trễ pha...")
    df_plotly = pd.DataFrame(predictions_sample_dict)

    fig_line = go.Figure()
    fig_line.add_trace(go.Scatter(x=df_plotly['timestamp'], y=df_plotly['y_true'], mode='lines', name='Thực tế (y_true)', line=dict(color='black', width=2)))
    fig_line.add_trace(go.Scatter(x=df_plotly['timestamp'], y=df_plotly['pred_full'], mode='lines', name='A: Full (Full Features)', line=dict(color='blue', dash='dash')))
    fig_line.add_trace(go.Scatter(x=df_plotly['timestamp'], y=df_plotly['pred_no_lag1'], mode='lines', name='B: No Lag 1', line=dict(color='orange')))
    fig_line.add_trace(go.Scatter(x=df_plotly['timestamp'], y=df_plotly['pred_no_lag_rolling'], mode='lines', name='D: No Lag & Rolling (Weather+Time)', line=dict(color='red')))

    fig_line.update_layout(
        title='So Sánh Trễ Pha Giữa Các Biến Thể Dự Báo vs Thực Tế tại 1 Trạm Mẫu',
        xaxis_title='Thời gian', yaxis_title='Sản lượng (kWh)',
        hovermode='x unified', template='plotly_white'
    )
    fig_line.show()

# 3. Bar chart Feature Importance của Biến thể D (no_lag_rolling)
v_dir_d = f'{OUTPUT_DIR}/no_lag_rolling'
if os.path.exists(f'{v_dir_d}/model_h1.pkl'):
    with open(f'{v_dir_d}/model_h1.pkl', 'rb') as f:
        model_d = pickle.load(f)
    v_feats_d = get_variant_features('no_lag_rolling', selected_features)
    imp_d = pd.DataFrame({'feature': v_feats_d, 'importance': model_d.feature_importances_})
    imp_d = imp_d.sort_values('importance', ascending=False).head(10)

    plt.figure(figsize=(9, 4))
    plt.barh(imp_d['feature'], imp_d['importance'], color='firebrick')
    plt.xlabel('Feature Importance (Split count)')
    plt.title('Top 10 Đặc Trưng Quan Trọng Nhất Của Biến Thể D (Khi Không Có Lag/Rolling)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 9. Export kết quả ra CSV

In [ ]:
# Export ablation_metrics.csv
csv_path = f'{OUTPUT_DIR}/ablation_metrics.csv'
df_ablation.to_csv(csv_path, index=False)

print("Đã xuất thành công báo cáo Ablation Metrics tại: " + csv_path)
print("Danh sách các file mô hình đã lưu tại " + OUTPUT_DIR + ":")
for v in VARIANTS:
    print("- " + OUTPUT_DIR + "/" + v + "/model_h1.pkl & model_h4.pkl")